## Imports and Setup

In [1]:
from pynwb import NWBHDF5IO
import os,glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import pickle
from scipy.ndimage import gaussian_filter1d
from scipy import stats
from pathlib import Path
# Ignore specific warning about cached namespaces
warnings.filterwarnings("ignore", message="Ignoring cached namespace")
warnings.filterwarnings("ignore", category=DeprecationWarning)


## Configuration and Paths

In [2]:
bin_size = 0.1 # for estimating firing rate
sigma = 1 # for smoothing
preGoCue = 1
postGoCue = 2

In [3]:
DATA_ROOT = "./data"
RAW_DIR = f"{DATA_ROOT}/raw"
PROCESSED_DIR = f"{DATA_ROOT}/processed"

nwb_file = "sub-M_ses-CO-20140303_behavior+ecephys.nwb"
dataset_name = Path(nwb_file).stem
# processed_folder = f"{PROCESSED_DIR}/{dataset_name}"
# os.makedirs(processed_folder, exist_ok=True)

config_name = f"binsize{int(bin_size*1000)}ms_sigma{int(bin_size*sigma*1000)}ms_pre{preGoCue}_post{postGoCue}"
processed_folder = f"{PROCESSED_DIR}/{dataset_name}/{config_name}"
os.makedirs(processed_folder, exist_ok=True)



In [4]:
io = NWBHDF5IO(f"{RAW_DIR}/{nwb_file}", mode="r")
nwbfile = io.read()
trials_df = nwbfile.trials.to_dataframe()
num_trials = trials_df.shape[0]

## Extract Position Data

In [5]:
units = nwbfile.units
units_df = units.to_dataframe()
units_df.head()

pos = np.array(nwbfile.processing["behavior"].data_interfaces["Position"].spatial_series["cursor_pos"].data)
ts = np.array(nwbfile.processing["behavior"].data_interfaces["Position"].spatial_series["cursor_pos"].timestamps)
pos.shape,ts.shape

((105359, 2), (105359,))

## Extract Neural Data (M1, PMd)

In [6]:
# Define a function to compute the firing rate
def compute_firing_rate(spike_times, bin_size=0.1, max_time=None, sigma= 0.5):
    if max_time is None:
        # Calculate the maximum time (end of the recording)
        max_time = np.max(spike_times)
    
    # Create time bins based on the bin size (e.g., 0.1s for 100ms)
    time_bins = np.arange(0, max_time + bin_size, bin_size)
    
    # Digitize the spike times into the time bins
    spike_counts, _ = np.histogram(spike_times, bins=time_bins)
    
    # Calculate firing rate (spikes per second)
    firing_rate = spike_counts / bin_size  # Divide by bin size to get the rate in Hz
    
    # Apply Gaussian smoothing
    smoothed_firing_rate = gaussian_filter1d(firing_rate, sigma=sigma)
    
    return smoothed_firing_rate, time_bins[:-1]  # Return the firing rate and time bin centers


# Initialize a dictionary to store new columns
spike_times_columns = {}
pos_by_trial = []


for electrode in range(len(units_df)):
    # Get spike times and region for each electrode
    spike_times = units_df["spike_times"].iloc[electrode]
    region = units_df["electrodes"].iloc[electrode].group_name.iloc[0].replace("electrode_group_", "")
    
    # Create a list to store spike times per trial for the current electrode
    spike_times_by_trial = []

    # Iterate over each trial
    for idx, row in trials_df.iterrows():
        if row["result"]!="R":
            spike_times_by_trial.append([])
            if electrode==0:
                pos_by_trial.append([])
        else:

            start_time = row['go_cue_time']-preGoCue
            stop_time = row['go_cue_time']+postGoCue

            # Select spike times for the current trial
            spikes_in_trial = spike_times[(spike_times >= start_time) & (spike_times < stop_time)]-start_time
            
            if electrode==0:
                time_indices = (ts >= start_time) & (ts < stop_time)
                pos_by_trial.append(pos[time_indices])
                

            # Compute the firing rate
            firing_rate, _ = compute_firing_rate(spikes_in_trial, bin_size, max_time=3, sigma=sigma) #stop_time-start_time

            # Append spike times for the current trial
    #         spike_times_by_trial.append(spikes_in_trial)
            spike_times_by_trial.append(firing_rate)
    
    # Store the list in the dictionary with the appropriate column name
    column_name = f'spike_times_{electrode}_{region}'
    spike_times_columns[column_name] = spike_times_by_trial


# Convert the dictionary to a DataFrame
spike_times_df = pd.DataFrame(spike_times_columns)

# Concatenate with trials_df along columns (axis=1)
trials_df = pd.concat([trials_df.reset_index(drop=True), spike_times_df.reset_index(drop=True)], axis=1)
trials_df['pos'] = pos_by_trial


# Display the DataFrame to verify
trials_df


,start_time,stop_time,target_on_time,go_cue_time,target_id,target_corners,target_dir,result,spike_times_0_M1,spike_times_1_M1,...,spike_times_109_PMd,spike_times_110_PMd,spike_times_111_PMd,spike_times_112_PMd,spike_times_113_PMd,spike_times_114_PMd,spike_times_115_PMd,spike_times_116_PMd,spike_times_117_PMd,pos
0,10.564,11.749,NaN,NaN,NaN,"[nan, nan, nan, nan]",NaN,A,[],[],...,[],[],[],[],[],[],[],[],[],[]
1,12.001,30.870,NaN,NaN,NaN,"[nan, nan, nan, nan]",NaN,A,[],[],...,[],[],[],[],[],[],[],[],[],[]
2,31.121,32.620,31.895267,NaN,7.0,"[-6.672598838806152, 6.64106559753418, -4.6725...",2.356194,A,[],[],...,[],[],[],[],[],[],[],[],[],[]
3,32.872,43.243,40.851500,41.729700,2.0,"[-6.672598838806152, 6.64106559753418, -4.6725...",0.000000,R,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",...,"[15.964908383992563, 18.606079587483478, 17.21...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[9.414431803346496, 7.533290314741386, 5.42499...","[0.0013383062461474176, 0.044318616200312654, ...","[0.0, 0.0, 0.0013383062461474176, 0.0469952286...","[12.865293528946578, 6.503481351953459, 4.1294...","[22.513704913951223, 23.274076373278284, 26.14...","[0.0, 0.0, 0.0, 0.0013383062461474176, 0.04431...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[[-0.7011939091924049, -0.44669855081781407], ..."
4,43.495,47.350,44.755200,45.936133,1.0,"[4.654601573944092, 6.659106254577637, 6.65460...",0.785398,R,"[0.0013383062461474176, 0.044318616200312654, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.001338306246147417...",...,"[1.1697980870608609, 4.8891007543169165, 8.607...","[2.9596257307730514, 4.03375330976129, 2.42105...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.002676612492294835, 0.0886372324006253...","[25.732267108580558, 19.414431803346496, 11.21...","[0.0, 0.0013383062461474176, 0.044318616200312...","[19.227447450380954, 8.878877192319154, 1.7580...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.00133830...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[[0.39074599320831105, -0.3718896253638633], [..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
228,1030.621,1034.866,1032.400200,1033.547033,6.0,"[-8.999977111816406, 0.9808881878852844, -6.99...",3.141593,R,"[19.227447450380954, 8.878877192319154, 1.7526...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0013383062461...",...,"[6.410487456373133, 3.003944346973364, 1.12414...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[7.133026420366017, 7.090387854853111, 12.9569...","[0.5869065028996516, 2.512366607705075, 5.1578...","[15.82491925971963, 10.539911274207045, 6.6391...","[0.0, 0.0, 0.0013383062461474176, 0.0443186162...","[47.87066501035456, 25.335363315580008, 9.4702...","[6.409149150126985, 2.9596257307730514, 0.5842...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[[0.1933812426920305, 0.3468933384771873], [0...."
229,1035.118,1038.708,1036.655767,1037.573033,0.0,"[-1.0, 9.0, 1.0, 7.0]",1.570796,R,"[0.6298868128538169, 2.9609640370191985, 6.409...","[0.0, 0.0, 0.0, 0.0, 0.0013383062461474176, 0....",...,"[0.7225389639928844, 4.086443507879746, 11.832...","[0.0, 0.0013383062461474176, 0.044318616200312...","[3.6351694660733282, 7.534628620987534, 11.249...","[0.09265215113906757, 1.1254794708605482, 5.42...","[16.36616884017282, 12.509689995212781, 9.2913...","[0.04565692244646007, 0.5399112742070441, 2.41...","[12.612199402554335, 22.96665900644505, 40.970...","[0.6298868128538169, 2.9609640370191985, 6.409...","[0.5842298904073567, 2.4210527628121548, 3.989...","[[0.619330707981506, -0.07566977306396439], [0..."
230,1038.959,1042.500,1040.674733,1041.267233,4.0,"[-0.987258791923523, -6.999989986419678, 1.012...",-1.570796,R,"[0.0, 0.0, 0.0013383062461474176, 0.0443186162...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",...,"[6.409149150126985, 2.9596257307730514, 0.5855...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.00133830...","[3.0522778819121186, 5.1

## Align Trials to Go Cue

In [7]:
direction = []
trial_times = []
for trial_idx in range(trials_df.shape[0]):
    if trials_df.iloc[trial_idx].result=="R":
        direction.append([trials_df["target_id"].iloc[trial_idx],trials_df["target_dir"].iloc[trial_idx]])
        trial_times.append([trials_df["start_time"].iloc[trial_idx],trials_df["go_cue_time"].iloc[trial_idx],trials_df["stop_time"].iloc[trial_idx]])
direction = np.array(direction)
trial_times = np.array(trial_times)

# Step 1: Fecth position data
position_data = []

for trial_idx in range(trials_df.shape[0]):
    if trials_df.iloc[trial_idx].result=="R":
        position_data.append(trials_df['pos'].iloc[trial_idx])

position_data = np.array(position_data)



data_M1 = []
# Step 1: Select only the neuron columns
neuron_columns = [col for col in trials_df.columns if (col.startswith("spike_times") and col.endswith("M1"))]

for neuron in neuron_columns:
    neuron_data = []
    for trial_idx in range(trials_df.shape[0]):
        if trials_df.iloc[trial_idx].result=="R":
            neuron_data.append(trials_df[neuron].iloc[trial_idx])
    data_M1.append(neuron_data)

data_M1 = np.array(data_M1)


data_PMd = []
neuron_columns = [col for col in trials_df.columns if (col.startswith("spike_times") and col.endswith("PMd"))]
for neuron in neuron_columns:
    neuron_data = []
    for trial_idx in range(trials_df.shape[0]):
        if trials_df.iloc[trial_idx].result=="R":
            neuron_data.append(trials_df[neuron].iloc[trial_idx])
    data_PMd.append(neuron_data)

data_PMd = np.array(data_PMd)
position_data.shape,data_M1.shape,data_PMd.shape,direction.shape,trial_times.shape


((209, 300, 2), (52, 209, 30), (66, 209, 30), (209, 2), (209, 3))

In [8]:
np.save(f"{processed_folder}/position.npy", position_data)

np.save(f"{processed_folder}/M1_rates.npy", data_M1)
np.save(f"{processed_folder}/PMd_rates.npy", data_PMd)

np.save(f"{processed_folder}/condition.npy", direction)
np.save(f"{processed_folder}/trial_times.npy", trial_times)

